[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/gaussian/gaussian.ipynb)

# A Gaussian from Point Moments

This example fits a Gaussian to a cloud of points and draws its one-sigma ellipsoid. Usually that takes two steps: find the mean, then the spread of the points about it, measured along the cloud's own axes. Here neither the centre nor the axes are found first. One average over the points holds both the cloud's location and its spread, and its inverse measures, at any point of the plane, how many standard deviations away from the cloud it lies.

Points are antivectors of projective geometric algebra and planes vectors, and each point's dyad, `point * (Plane & point)`, is a map from planes to points. Averaged over the cloud it is the second moment, homogeneous, so it holds the mean as well; its inverse, the precision, is a map from points to planes, and at a unit-weight point it gives one plus the squared Mahalanobis distance, `precision(point) & point`. The Gaussian is its exponential, and the one-sigma ellipsoid is a quadric, the precision less twice the dyad of the plane of unit weight, which the precision itself recovers from the data.

In the notation of statistics the second moment reads as the augmented matrix $\begin{pmatrix}\Sigma + \boldsymbol\mu\boldsymbol\mu^{\mathsf T} & \boldsymbol\mu \\ \boldsymbol\mu^{\mathsf T} & 1\end{pmatrix}$, whose inverse evaluates at $(\mathbf x, 1)$ to $1 + (\mathbf x - \boldsymbol\mu)^{\mathsf T}\Sigma^{-1}(\mathbf x - \boldsymbol\mu)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext
from numga.algebras import PGA2D as ga

from examples.util.animation import save_figure
from examples.quadrics.gaussian import render

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                      # Antivector-
Plane = ga.gatype.vector()                          # Vector+
Scalar = ga.gatype.scalar()                         # Scalar+
# A map from points to planes: a precision, or a quadric.
Polarity = ga.gatype((Plane, Point))

## 1. A stretched, moved cloud

The cloud is drawn stretched along x, then turned and moved as a batch, so that neither its centre nor its axes are where the coordinates are.

In [ ]:
# The cloud's size, its spread along x and y before it is moved, and the seed of its samples; how it is
# moved, a translation and a turn as generator coefficients.
n_points, spread, seed = 400, np.array([1.5, 0.5]), 4
shift, turn = (0.6, 0.3), 0.3


In [ ]:
placement = (mv.xw * shift[0] - mv.yw * shift[1]).exp() * (mv.xy * turn).exp()   # [] Motor+
points = placement >> mv.point(np.random.default_rng(seed).normal(size=(n_points, 2)) * spread)   # [points] Point-


## 2. The fit

The mean of the points' dyads is the second moment, and its inverse the precision. At a unit-weight point the precision gives one plus the squared Mahalanobis distance, so the Gaussian, with peak one, is the exponential of minus half of what is left. The plane of unit weight is the precision's image of the cloud's mean, the one plane that pairs to one with every unit-weight point; subtracting its dyad twice from the precision moves the zero of the form to Mahalanobis distance one: the one-sigma ellipsoid as a quadric, a map from points to planes.

In [ ]:
def fit_gaussian(points: Point) -> tuple[Polarity, Polarity]:
    """The precision of unit-weight points comes with their 1σ ellipsoid, a quadric: both maps from points to planes."""
    # Each point contributes its dyad, a rank-one map from planes to points; their mean holds both the cloud's
    # location and its spread, without centring.
    moment = (points * (Plane & points)).mean(axis=0)                          # [] Point- <- Plane+
    precision = moment.inverse()                                               # [] Plane+ <- Point-
    # The plane of unit weight, recovered from the data: weight & point == 1 for every unit-weight point. Its dyad
    # evaluates to that constant squared; subtracted twice, it makes Mahalanobis distance one the zero locus.
    weight = precision(points.mean(axis=0))                                    # [] Plane+
    return precision, precision - 2 * weight * (weight & Point)


def gaussian(samples: Point) -> Scalar:
    """The Gaussian at unit-weight points has peak one: the precision evaluates to one plus the squared Mahalanobis
    distance, and the Gaussian is the exponential of minus half of the rest."""
    squared_distance = (precision(samples) & samples) - 1                      # [...] Scalar+
    return (-0.5 * squared_distance).exp()


precision, ellipsoid = fit_gaussian(points)                                    # [] Plane+ <- Point-, Plane+ <- Point-
save_figure(render.draw_gaussian(points, gaussian, ellipsoid), "gaussian");


In [ ]:
# At the cloud's mean: peak density one, and the ellipsoid's form minus one, inside it.
mean = points.mean(axis=0)                                                     # [] Point-
print("density at the mean:", gaussian(mean).to_array(), "  ellipsoid's form at the mean:", (ellipsoid(mean) & mean).to_array())


In [ ]:
# checks
# The density peaks at one at the cloud's mean, and the ellipsoid's form is the squared Mahalanobis distance less
# one, so the ellipsoid is the contour where the density falls to exp(-1/2).
np.testing.assert_allclose(gaussian(mean).to_array(), 1.0, atol=1e-12)
np.testing.assert_allclose((ellipsoid(points) & points).to_array(), -2.0 * np.log(gaussian(points).to_array()) - 1.0, atol=1e-8)
